# Chapter 2: The Road Network

This notebook pulls the drivable road network from OpenStreetMap using OSMnx, cleans the node and edge data and loads it into Neo4j Aura as `Intersection` nodes connected by `ROAD` relationships.

## 1. Install Dependencies

In [1]:
%pip install geopandas==1.1.4 \
             neo4j==5.28.1 \
             networkx==3.6.1 \
             numpy==2.2.6 \
             osmnx==2.1.1 \
             pandas==3.0.3 \
             pyyaml==6.0.3 \
             shapely==2.1.2 \
             tqdm==4.70.0 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


In [2]:
%pip install pyrosm==0.13.1 --no-deps --quiet

print("pyrosm installed.")

Note: you may need to restart the kernel to use updated packages.
pyrosm installed.


## 2. Imports

In [3]:
import networkx as nx
import os
import osmnx as ox
import pandas as pd

from config_validator import load_config, ConfigError
from neo4j import GraphDatabase
from pyrosm import OSM
from tqdm.notebook import tqdm

## 3. Configuration

In [4]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

try:
    cfg = load_config("config.yaml")
except (FileNotFoundError, ConfigError) as e:
    raise SystemExit(f"Config error: {e}")

PLACE        = cfg["city"]["osmnx_place"]
NETWORK_TYPE = cfg["city"]["network_type"]
BATCH_SIZE   = 500

print(f"City : {cfg['city']['name']}")
print(f"Place: {PLACE}")
print("Credentials set.")

City : London Borough of Merton
Place: London Borough of Merton, UK
Credentials set.


## 4. Pull Road Network From OpenStreetMap

In [5]:
if "osm_file" in cfg["city"] and cfg["city"]["osm_file"]:
    osm_path = cfg["city"]["osm_file"]
    if not os.path.exists(osm_path):
        raise FileNotFoundError(
            f"OSM file not found: {osm_path}\n"
            f"Run 00_prepare_osm.ipynb first to generate this file."
        )
    osm = OSM(osm_path)
    nodes_gdf, edges_gdf = osm.get_network(network_type="driving", nodes=True)
    G = osm.to_graph(nodes_gdf, edges_gdf, graph_type="networkx")
elif isinstance(PLACE, list):
    graphs = [ox.graph_from_place(p, network_type=NETWORK_TYPE) for p in PLACE]
    G = graphs[0]
    for g in graphs[1:]:
        G = nx.compose(G, g)
else:
    G = ox.graph_from_place(PLACE, network_type=NETWORK_TYPE)

print(f"Nodes : {len(G.nodes):,}")
print(f"Edges : {len(G.edges):,}")

Nodes : 3,203
Edges : 7,317


## 5. Extract and Clean Node and Edge Data

In [6]:
def flatten(val):
    """Return first element if value is a list, otherwise return as-is."""
    if isinstance(val, list):
        return val[0]
    return val

def parse_maxspeed(val):
    """Extract integer mph from strings like '50 mph' or '50'. Returns None if unparseable."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    val = flatten(val)
    try:
        return int(str(val).replace(" mph", "").strip())
    except (ValueError, AttributeError):
        return None

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Nodes
nodes_clean = nodes_gdf[["y", "x", "street_count"]].reset_index()
nodes_clean = nodes_clean.rename(columns={"osmid": "node_id", "y": "lat", "x": "lon"})

# Edges
edges_clean = edges_gdf[["osmid", "name", "highway", "maxspeed", "length", "oneway"]].reset_index()[["u", "v", "osmid", "name", "highway", "maxspeed", "length", "oneway"]]
edges_clean["osmid"]    = edges_clean["osmid"].apply(flatten).astype(str)
edges_clean["name"]     = edges_clean["name"].apply(flatten)
edges_clean["highway"]  = edges_clean["highway"].apply(flatten)
edges_clean["maxspeed"] = edges_clean["maxspeed"].apply(parse_maxspeed)
edges_clean["length_m"] = edges_clean["length"].round(2)
edges_clean = edges_clean.drop(columns=["length"])

# Deduplicate edges -- OSMnx assigns the same osmid to multiple segments
# in two cases: bidirectional roads (u->v and v->u) and segmented OSM ways.
# Sort by length descending and keep the longest segment per osmid+u+v pair.
edges_clean = edges_clean.sort_values('length_m', ascending=False).drop_duplicates(subset=['osmid', 'u', 'v'], keep='first')

print(f"Nodes : {len(nodes_clean):,}")
print(f"Edges : {len(edges_clean):,} (after deduplication)")
print()
print("Sample nodes:")
print(nodes_clean.head(3).to_string(index=False))
print()
print("Sample edges:")
print(edges_clean.head(3).to_string(index=False))

Nodes : 3,203
Edges : 7,276 (after deduplication)

Sample nodes:
 node_id       lat       lon  street_count
  292340 51.402526 -0.241188             3
  292344 51.405640 -0.241719             3
11080838 51.395492 -0.158893             3

Sample edges:
       u        v      osmid           name     highway  maxspeed  oneway  length_m
26158160 26159316   23887810 Home Park Road residential      20.0   False   1013.41
26159316 26158160   23887810 Home Park Road residential      20.0   False   1013.41
11080848 11080841 1337691610   Croydon Road     primary      30.0   False    982.52


## 6. Connect to Neo4j

In [7]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity()) # None is expected
print("Neo4j connection verified.")

None
Neo4j connection verified.


## 7. Clear Existing Data

In [8]:
# Clear all existing data
with driver.session() as session:
    session.run("""
        MATCH (n)
        CALL (n) { DETACH DELETE n } IN TRANSACTIONS OF 1000 ROWS
    """)
    print("Database cleared.")

Database cleared.


## 8. Create Constraint and Index

In [9]:
with driver.session() as session:
    session.run("""
        CREATE CONSTRAINT intersection_node_id IF NOT EXISTS
        FOR (i:Intersection) REQUIRE i.node_id IS UNIQUE
    """)
    session.run("""
        CREATE POINT INDEX intersection_location IF NOT EXISTS
        FOR (i:Intersection) ON (i.location)
    """)

print("Constraint and index created")

Constraint and index created


## 9. Load Nodes

In [10]:
def load_nodes(tx, batch):
    tx.run("""
        UNWIND $rows AS row
        MERGE (i:Intersection {node_id: row.node_id})
        SET i.lat          = row.lat,
            i.lon          = row.lon,
            i.street_count = row.street_count,
            i.location     = point({latitude: row.lat, longitude: row.lon})
    """, rows=batch)

def batch_load(df, tx_func, label):
    rows = df.where(pd.notnull(df), None).to_dict("records")
    total = len(rows)
    for i in tqdm(range(0, total, BATCH_SIZE), desc=label, unit="batch"):
        batch = rows[i:i + BATCH_SIZE]
        with driver.session() as session:
            session.execute_write(tx_func, batch)
    print(f"{label}: done ({total:,} rows)")

batch_load(nodes_clean, load_nodes, "Nodes")

Nodes:   0%|          | 0/7 [00:00<?, ?batch/s]

Nodes: done (3,203 rows)


## 10. Load Edges

In [11]:
def load_edges(tx, batch):
    tx.run("""
        UNWIND $rows AS row
        MATCH (a:Intersection {node_id: row.u})
        MATCH (b:Intersection {node_id: row.v})
        MERGE (a)-[r:ROAD {osmid: row.osmid, u: row.u, v: row.v}]->(b)
        SET r.name     = row.name,
            r.highway  = row.highway,
            r.maxspeed = row.maxspeed,
            r.oneway   = row.oneway,
            r.length_m = row.length_m
    """, rows=batch)

batch_load(edges_clean, load_edges, "Edges")  # edges_clean is already deduped

Edges:   0%|          | 0/15 [00:00<?, ?batch/s]

Edges: done (7,276 rows)


## 11. Verify

In [12]:
with driver.session() as session:
    result = session.run("""
        MATCH (i:Intersection)
        RETURN count(i) AS intersections
    """)
    print(f"Intersections : {result.single()['intersections']:,}")

    result = session.run("""
        MATCH ()-[r:ROAD]->()  
        RETURN count(r) AS roads
    """)
    print(f"Roads         : {result.single()['roads']:,}")

    result = session.run("""
        MATCH ()-[r:ROAD]->() 
        WHERE r.name IS NOT NULL
        RETURN r.name AS name, r.highway AS highway, r.maxspeed AS maxspeed_mph, r.length_m AS length_m
        ORDER BY r.length_m DESC
        LIMIT 5
    """)
    print()
    print("Longest named roads:")
    for rec in result:
        print(f"  {rec['name']:<30} {rec['highway']:<12} {str(rec['maxspeed_mph']) + ' mph':<10} {rec['length_m']} m")

Intersections : 3,203
Roads         : 7,276

Longest named roads:
  Home Park Road                 residential  20.0 mph   1013.41 m
  Home Park Road                 residential  20.0 mph   1013.41 m
  Croydon Road                   primary      30.0 mph   982.52 m
  Croydon Road                   primary      30.0 mph   982.52 m
  Croydon Road                   primary      30.0 mph   865.69 m


## 12. Teardown

In [13]:
driver.close()
print("Driver closed.")

Driver closed.
